# Hướng dẫn fine-tune cho IDP hóa đơn tiếng Việt

Notebook này là blueprint reproducible cho MC-OCR/VietOCR/LayoutLMv3. Không tự tải model hay chạy GPU khi mở; thay các đường dẫn và bật từng cell trong môi trường train riêng.

## 1. Chuẩn hóa dataset

Mỗi ảnh có annotation JSON gồm `text`, `tokens`, `boxes` và `entities`. Tọa độ phải ở pixel của ảnh sau cùng một bước resize/deskew dùng cho train. Chia train/validation theo **hóa đơn**, không chia các crop của cùng ảnh sang hai tập.

```json
{"image": "images/0001.jpg", "tokens": ["Cà phê", "2", "30.000"], "boxes": [[20,100,220,130], [300,100,330,130], [500,100,600,130]], "entities": [{"label": "SELLER", "token_indices": [0]}]}
```

In [ ]:
from pathlib import Path
import json

DATA_ROOT = Path("data/mc_ocr")
ANNOTATIONS = DATA_ROOT / "annotations.jsonl"
rows = [json.loads(line) for line in ANNOTATIONS.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"samples={len(rows)}")
print(rows[0].keys() if rows else "empty")

## 2. Preprocessing thống nhất

Dùng `backend.app.idp_engine.preprocessor.ImagePreprocessor` để tạo `enhanced_gray` và metadata. Khi train LayoutLM, map box từ ảnh đã xử lý; khi train VietOCR, dùng crop polygon sau perspective warp. Không trộn box của ảnh gốc với ảnh đã warp.

Khuyến nghị augmentation nhẹ: xoay ±2°, brightness/contrast, Gaussian noise và blur. Không dùng crop mạnh làm mất phần tổng tiền hoặc mã hóa đơn.

In [ ]:
# Chạy từ root repository sau khi cài requirements.
from backend.app.idp_engine.preprocessor import ImagePreprocessor

processor = ImagePreprocessor()
sample = processor.process(DATA_ROOT / rows[0]["image"])
sample.metadata.model_dump()

## 3. Fine-tune VietOCR

Tạo manifest crop: mỗi dòng `{"image": "crops/0001_03.png", "text": "Cà phê sữa"}`. Load `vgg_transformer`, thay decoder vocabulary nếu cần, freeze encoder ở epoch đầu, dùng early stopping trên CER validation. Theo dõi CER/WER của text và không chỉ loss.

Pseudo-steps: `crop -> normalize height -> teacher forcing -> cross entropy -> validation CER`. Giữ nguyên dấu tiếng Việt trong vocabulary; không lowercase nếu label nghiệp vụ phân biệt mã hóa đơn.

## 4. LayoutLMv3/KIE

Đưa OCR tokens, normalized boxes `[0, 1000]`, image và BIO labels vào processor. Labels đề xuất: `O`, `B/I-SELLER`, `B/I-ADDRESS`, `B/I-TIMESTAMP`, `B/I-INVOICE_ID`, `B/I-LINE_ITEMS`, `B/I-TAX`, `B/I-TOTAL_COST`. Line items nên có một bước hậu xử lý để group theo dòng và parse quantity/unit price/amount.

Đánh giá tách biệt: token classification F1, entity exact-match F1, và công thức tổng tiền sau normalization.

In [ ]:
def normalize_box(box, width, height):
    x1, y1, x2, y2 = box
    return [round(1000*x1/width), round(1000*y1/height),
            round(1000*x2/width), round(1000*y2/height)]

# Sau khi model dự đoán, ghi JSONL theo format evaluation/evaluate_metrics.py
# {"reference": {"text": ..., "entities": ...},
#  "prediction": {"text": ..., "entities": ...}}

## 5. Reproducibility checklist

- Lưu seed, version model, preprocessing config và train/val/test manifest.
- Không dùng ảnh test trong prompt/few-shot của VLM.
- Báo cáo CER, WER, entity Precision/Recall/F1, math-pass rate và tỷ lệ `NEEDS_REVIEW`.
- So sánh cùng một tập test giữa baseline 2-stage và VLM; ghi latency và chi phí API nếu dùng cloud.